# DR stage detection - Kaggle runner
Clones the project repo, runs the selected notebooks on GPU and pushes results back to GitHub.
Prereqs: dataset attached, GPU on, internet on, secret `GITHUB_TOKEN` (classic PAT, `repo` scope).
Use **Save Version -> Save & Run All** so it keeps running when the browser tab is closed.

In [ ]:
# 1. Bootstrap: clone repo (token only used for the clone, then removed from git config)
import os, subprocess, shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO = "https://github.com/pradeesha999/dr-stage-detection-v2.git"
os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
dst = Path("/kaggle/working/dr_project")
if not dst.exists():
    subprocess.run(["git", "clone", "-q", REPO.replace("https://", f"https://{os.environ['GITHUB_TOKEN']}@"), str(dst)], check=True)
    subprocess.run(["git", "-C", str(dst), "remote", "set-url", "origin", REPO], check=True)
print(subprocess.run(["git", "-C", str(dst), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Recover results of a previous run: add that version's output as an Input
# (Add Input -> Your Work -> this notebook). Models, metrics, figures and the
# executed notebooks are copied into the fresh clone.
for prev in Path("/kaggle/input").glob("**/dr_project"):
    if prev.is_dir() and (prev / "outputs").exists():
        print("recovering from", prev)
        for sub in ("outputs/models", "outputs/metrics", "outputs/figures"):
            if (prev / sub).exists():
                shutil.copytree(prev / sub, dst / sub, dirs_exist_ok=True)
        for nb in (prev / "notebooks").glob("0*_*.ipynb"):
            shutil.copy(nb, dst / "notebooks" / nb.name)
        break
print(sorted(p.name for p in (dst / "outputs" / "models").glob("*")))


In [ ]:
# 2. Which notebooks to run. First full run: "01 02 03 04 05".
#    Recovery / evaluation-only run (previous output attached as Input): "01 02 03 05"
NOTEBOOKS = "01 02 03 05"
!cd /kaggle/working/dr_project && bash scripts/run_notebooks.sh {NOTEBOOKS}


In [ ]:
# 3. Push executed notebooks, figures, metrics back to GitHub
!cd /kaggle/working/dr_project && bash scripts/push_results.sh

In [ ]:
# 4. Keep the trained model as a Kaggle output too (models are git-ignored: too large)
!ls -lh /kaggle/working/dr_project/outputs/models/ /kaggle/working/dr_project/outputs/metrics/ 2>/dev/null | head -40